# 04 — Tableau extracts (Phase 4)

**Purpose.** Produce the seven CSV extracts the six Phase 4 Tableau
workbooks consume, plus the `region_coordinates.csv` reference file.
Apply the small derivations the silver-layer parquets defer to the
visualisation layer (Component 5's 100B-energy correction, Component 2's
unmatched flag, Component 8's flow-shape reshape). No new analytical
claims; this is a presentation-layer extract notebook.

**Inputs.** The two silver-layer parquets under
`../data/processed/integrated/`:

- `models_enriched.parquet` (194 rows × 105 columns)
- `regions_geographic.parquet` (37 rows × 12 columns)

**Outputs.**

- `../data/processed/tableau/c4_trajectory.csv`
- `../data/processed/tableau/c3_regions.csv`
- `../data/processed/tableau/c2_model_energy.csv`
- `../data/processed/tableau/c8_flow.csv`
- `../data/processed/tableau/c6_small_multiples.csv`
- `../data/processed/tableau/c5_aggregate_scaling.csv`
- `../data/reference/region_coordinates.csv`

## A. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

INTEGRATED = Path("../data/processed/integrated")
TABLEAU = Path("../data/processed/tableau")
REFERENCE = Path("../data/reference")
TABLEAU.mkdir(parents=True, exist_ok=True)
REFERENCE.mkdir(parents=True, exist_ok=True)

models_enriched = pd.read_parquet(INTEGRATED / "models_enriched.parquet")
regions_geographic = pd.read_parquet(INTEGRATED / "regions_geographic.parquet")
print(f"models_enriched:    {models_enriched.shape}")
print(f"regions_geographic: {regions_geographic.shape}")

TIER_LABEL = {300: "short", 1000: "medium", 1500: "long"}

models_enriched:    (194, 105)
regions_geographic: (37, 12)


In [2]:
# Adapted from notebooks/01_ingestion.ipynb / 02_integration.ipynb.
# Copied (not imported) so this notebook is self-contained.
def validate_and_write(df, path, expected_min_rows, must_have_columns):
    missing = [c for c in must_have_columns if c not in df.columns]
    assert not missing, f"{path.name}: missing required columns {missing}"
    assert len(df) >= expected_min_rows, (
        f"{path.name}: {len(df)} rows < expected min {expected_min_rows}"
    )
    print(f"{path.relative_to(path.parent.parent.parent)}: "
          f"rows={len(df)} cols={len(df.columns)}")
    df.to_csv(path, index=False, encoding="utf-8")
    return path

## B. `c4_trajectory.csv`

Component 4 — training compute trajectory. The 15 frontier models with
disclosed `Training compute (FLOP)` and `Publication date`. Long-tier
spine, deduplicated on `Model` (each model has one long-tier row).

In [3]:
c4 = (
    models_enriched[
        (models_enriched["Query Length"] == 1500)
        & models_enriched["Training compute (FLOP)"].notna()
        & models_enriched["Publication date"].notna()
    ]
    .drop_duplicates("Model")
    [[
        "Model",
        "Publication date",
        "Organization",
        "Training compute (FLOP)",
        "Parameters",
        "Mean Combined Energy (Wh)",
    ]]
    .sort_values("Publication date")
    .reset_index(drop=True)
)
validate_and_write(
    c4,
    TABLEAU / "c4_trajectory.csv",
    expected_min_rows=15,
    must_have_columns=[
        "Model",
        "Publication date",
        "Organization",
        "Training compute (FLOP)",
        "Parameters",
        "Mean Combined Energy (Wh)",
    ],
)

processed\tableau\c4_trajectory.csv: rows=15 cols=6


WindowsPath('../data/processed/tableau/c4_trajectory.csv')

## C. `c3_regions.csv`

Component 3 — global cloud-region map. All 12 columns of
`regions_geographic.parquet` written verbatim, no filtering.
Tableau joins this to `region_coordinates.csv` on `region_name` for
geocoding.

In [4]:
c3 = regions_geographic.copy()
validate_and_write(
    c3,
    TABLEAU / "c3_regions.csv",
    expected_min_rows=37,
    must_have_columns=[
        "provider",
        "region_name",
        "iso3",
        "state_abb",
        "wue_site",
        "cif_site",
        "wue_source",
        "cif_source",
        "grid_carbon_intensity_gco2_kwh",
        "water_stress_score",
        "us_facility_count_in_state",
        "us_facility_sqft_total_in_state",
    ],
)

processed\tableau\c3_regions.csv: rows=37 cols=12


WindowsPath('../data/processed/tableau/c3_regions.csv')

## D. `region_coordinates.csv`

Hand-curated reference table mapping each of the 37 cloud regions in
`regions_geographic.parquet` to public-infrastructure city-level
(latitude, longitude) coordinates. Sources: AWS Regional Services
List and Azure global-infrastructure pages. Data centres' exact
coordinates are not published; city-level placement is sufficient for
the Phase 4 map. Lives in `data/reference/`, not
`data/processed/tableau/`, because it is a hand-curated reference
table rather than a notebook-derived extract.

In [5]:
region_coordinates = {
    # AWS — 10 regions
    "Australia (Sydney)":           (-33.8688, 151.2093),
    "Canada (Central) (Ontario)":   (43.6532, -79.3832),
    "Europe (Frankfurt)":           (50.1109, 8.6821),
    "Europe (Ireland)":             (53.3498, -6.2603),
    "Europe (London)":              (51.5074, -0.1278),
    "Middle East (UAE)":            (24.4539, 54.3773),
    "South America (S\u00e3o Paulo)":    (-23.5505, -46.6333),
    "US East (Northern Virginia)":  (38.9469, -77.4903),
    "US East (Ohio)":               (40.0946, -82.7541),
    "US West (Oregon)":             (45.5152, -122.6784),
    # Azure — 27 regions
    "Australia East":               (-33.8688, 151.2093),
    "Brazil South":                 (-23.5505, -46.6333),
    "Canada Central":               (43.6532, -79.3832),
    "Central India":                (18.5204, 73.8567),
    "East Asia":                    (22.2783, 114.1747),
    "East US (PJM)":                (37.3719, -79.8164),
    "East US 2 (PJM)":              (37.3719, -79.8164),
    "France Central":               (48.8566, 2.3522),
    "Germany West Central":         (50.1109, 8.6821),
    "Italy North":                  (45.4642, 9.1900),
    "Japan East":                   (35.6762, 139.6503),
    "Korea Central":                (37.5665, 126.9780),
    "North Central US (MISO)":      (41.8781, -87.6298),
    "North Europe":                 (53.3498, -6.2603),
    "Qatar Central":                (25.2854, 51.5310),
    "South Africa North":           (-25.7479, 28.2293),
    "South Central US (ERCOT)":     (29.4241, -98.4936),
    "South East Asia":              (1.3521, 103.8198),
    "South India":                  (13.0827, 80.2707),
    "Spain Central":                (40.4168, -3.7038),
    "Sweden Central":               (59.3293, 18.0686),
    "Switzerland North":            (47.3769, 8.5417),
    "UAE North":                    (25.2854, 55.3219),
    "UK South":                     (51.5074, -0.1278),
    "West Europe":                  (52.3676, 4.9041),
    "West US (CISO)":               (37.7749, -122.4194),
    "West US 3 (APS)":              (33.4484, -112.0740),
}

# Validate the dictionary covers every region in regions_geographic.
covered = set(region_coordinates.keys())
actual = set(regions_geographic["region_name"].dropna().unique())
missing_from_dict = sorted(actual - covered)
extra_in_dict = sorted(covered - actual)
assert not missing_from_dict, (
    f"region_coordinates dict missing entries: {missing_from_dict}"
)
if extra_in_dict:
    print(f"note: dict contains regions not in regions_geographic: {extra_in_dict}")

rc = pd.DataFrame(
    [(r, lat, lon) for r, (lat, lon) in region_coordinates.items()],
    columns=["region_name", "latitude", "longitude"],
).sort_values("region_name").reset_index(drop=True)
validate_and_write(
    rc,
    REFERENCE / "region_coordinates.csv",
    expected_min_rows=37,
    must_have_columns=["region_name", "latitude", "longitude"],
)

data\reference\region_coordinates.csv: rows=37 cols=3


WindowsPath('../data/reference/region_coordinates.csv')

## E. `c2_model_energy.csv`

Component 2 — ranked dot plot, three small multiples by query-length
tier. Output is the cartesian product of all 66 models with all three
tiers (198 rows), with `is_unmatched=True` for the four rows where
energy is null at the long tier (the 2023-vintage models without
long-prompt benchmarks: GPT-3.5 Turbo, GPT-4, Llama 3 70B, Llama 3 8B).

Organization for Component 2 is `Company` (Jegham's per-model
attribution, populated for all 66 models), not Epoch's `Organization`
(populated only for the 134 matched-Epoch rows).

In [6]:
all_models = sorted(models_enriched["Model"].dropna().unique())
expected = pd.DataFrame(
    [(m, ql) for m in all_models for ql in [300, 1000, 1500]],
    columns=["Model", "Query Length"],
)
c2 = expected.merge(
    models_enriched[["Model", "Query Length", "Company", "Mean Combined Energy (Wh)"]],
    on=["Model", "Query Length"],
    how="left",
)

# Backfill Company from any tier when the long-tier row is missing.
company_lookup = (
    models_enriched.dropna(subset=["Company"])
    .drop_duplicates("Model")
    .set_index("Model")["Company"]
)
c2["Company"] = c2["Model"].map(company_lookup)

c2["tier"] = c2["Query Length"].map(TIER_LABEL)
c2["is_unmatched"] = (c2["tier"] == "long") & c2["Mean Combined Energy (Wh)"].isna()
c2 = c2.rename(columns={"Company": "Organization"})[[
    "Model",
    "Query Length",
    "tier",
    "Organization",
    "Mean Combined Energy (Wh)",
    "is_unmatched",
]]

n_unmatched = c2["is_unmatched"].sum()
unmatched_models = sorted(c2.loc[c2["is_unmatched"], "Model"].tolist())
print(f"is_unmatched=True rows: {n_unmatched} (expected 4)")
print(f"unmatched models: {unmatched_models}")
assert n_unmatched == 4, f"expected 4 unmatched rows, got {n_unmatched}"

validate_and_write(
    c2,
    TABLEAU / "c2_model_energy.csv",
    expected_min_rows=198,
    must_have_columns=[
        "Model",
        "Query Length",
        "tier",
        "Organization",
        "Mean Combined Energy (Wh)",
        "is_unmatched",
    ],
)

is_unmatched=True rows: 4 (expected 4)
unmatched models: ['GPT-3.5 Turbo', 'GPT-4', 'Llama 3 70B', 'Llama 3 8B']
processed\tableau\c2_model_energy.csv: rows=198 cols=6


WindowsPath('../data/processed/tableau/c2_model_energy.csv')

## F. `c8_flow.csv`

Component 8 — Sankey trace for one query along the GPT-4o (Aug) long
→ AWS US East (Northern Virginia) path. Six nodes, five flows. The
hard-coded values protect the Sankey from silent data shifts; an
assertion checks the upstream tables still match within 0.001.

In [7]:
gpt = models_enriched[
    (models_enriched["Model"] == "GPT-4o (Aug)")
    & (models_enriched["Query Length"] == 1500)
].iloc[0]
va = regions_geographic[
    regions_geographic["region_name"] == "US East (Northern Virginia)"
].iloc[0]

energy_actual = float(gpt["Mean Combined Energy (Wh)"])
carbon_actual = float(gpt["Mean Combined Carbon (gCO2e)"])
grid_actual = float(va["grid_carbon_intensity_gco2_kwh"])

TOL = 0.001
assert abs(energy_actual - 2.2215) < TOL, (
    f"GPT-4o (Aug) long energy drifted: {energy_actual} vs 2.2215"
)
assert abs(grid_actual - 327.17) < TOL, (
    f"Virginia grid intensity drifted: {grid_actual} vs 327.17"
)
assert abs(carbon_actual - 0.7553) < TOL, (
    f"GPT-4o (Aug) long carbon drifted: {carbon_actual} vs 0.7553"
)

c8 = pd.DataFrame(
    [
        ("1 query", "GPT-4o (Aug) inference", 2.2215, "Wh", "Per-query energy"),
        ("GPT-4o (Aug) inference", "AWS US East (Virginia)", 2.2215, "Wh", "Routed to region"),
        ("AWS US East (Virginia)", "Virginia grid", 2.2215, "Wh", "Drawn from grid"),
        ("Virginia grid", "Derived emissions", 0.7268, "gCO2e",
         "\u00d7 327.17 gCO2/kWh \u00f7 1000"),
        ("Derived emissions", "End", 0.7268, "gCO2e",
         "Net atmospheric carbon (vs Jegham 0.7553, -3.8%)"),
    ],
    columns=["source", "target", "value", "unit", "flow_label"],
)
validate_and_write(
    c8,
    TABLEAU / "c8_flow.csv",
    expected_min_rows=5,
    must_have_columns=["source", "target", "value", "unit", "flow_label"],
)

processed\tableau\c8_flow.csv: rows=5 cols=5


WindowsPath('../data/processed/tableau/c8_flow.csv')

## G. `c6_small_multiples.csv`

Component 6 — efficiency vs demand small multiples. The 12 selected
models from the Phase 3 lock × 3 query-length tiers = 36 rows. All
12 × 3 combinations are present (verified at the close of Phase 3).

In [8]:
c6_models = [
    "GPT-4o (Aug)",
    "Claude 3.7 Sonnet",
    "GPT-5 (minimal)",
    "GPT-5 (high)",
    "Llama 3.1 8B",
    "DeepSeek R1 (DeepSeek) [128k]",
    "Llama 3.2 1B",
    "GPT-5 nano (minimal)",
    "o3-pro",
    "Llama 3.1 405B Standard",
    "Claude 4 Opus",
    "Mistral Small (Feb)",
]
c6 = (
    models_enriched[models_enriched["Model"].isin(c6_models)]
    .assign(tier=lambda d: d["Query Length"].map(TIER_LABEL))
    [["Model", "Query Length", "tier", "Mean Combined Energy (Wh)"]]
    .sort_values(["Model", "Query Length"])
    .reset_index(drop=True)
)
missing_combos = []
for m in c6_models:
    for ql in [300, 1000, 1500]:
        if not ((c6["Model"] == m) & (c6["Query Length"] == ql)).any():
            missing_combos.append((m, ql))
assert not missing_combos, f"c6 missing combos: {missing_combos}"

validate_and_write(
    c6,
    TABLEAU / "c6_small_multiples.csv",
    expected_min_rows=36,
    must_have_columns=["Model", "Query Length", "tier", "Mean Combined Energy (Wh)"],
)

processed\tableau\c6_small_multiples.csv: rows=36 cols=4


WindowsPath('../data/processed/tableau/c6_small_multiples.csv')

## H. `c5_aggregate_scaling.csv`

Component 5 — aggregate scaling, long format with one row per
(Model, scale, metric).

**Diagnosis of the upstream 100B-MWh-energy defect.** The Phase 4
scoping pass surfaced that three columns in `models_enriched.parquet`
fail the published-vs-derived ratio audit at 100B scale: `Energy
Consumption of 100 Billion Prompts (MWh)`, `Household Energy Equiv. –
100B Prompts (MWh)`, and `University Energy Equiv. – 100B Prompts
(MWh)`. All three carry a 50× multiplier of the 1B-scale value rather
than the expected 100×, indicating a single 10× multiplier error in
Jegham's 100B-MWh-energy chain (the carbon and water columns at 100B
scale audit cleanly at the expected 100× ratio, as do all 50B columns
at the expected 50× ratio). This section computes derived 100B values
for all three MWh-energy columns and uses the derived values in the
extract; the published values are excluded.

In [9]:
long_tier = models_enriched[models_enriched["Query Length"] == 1500].copy()

# Apply the three derivations. The Energy Consumption column is
# derived from per-query Wh; the two equivalence columns are derived
# from their 1B-scale published value times 100.
long_tier["Energy Consumption of 100 Billion Prompts (MWh) DERIVED"] = (
    long_tier["Mean Combined Energy (Wh)"] * 100e9 / 1e6
)
long_tier["Household Energy Equiv. – 100B Prompts (MWh) DERIVED"] = (
    long_tier["Household Energy Equiv. – 1B Prompts (MWh)"] * 100
)
long_tier["University Energy Equiv. – 100B Prompts (MWh) DERIVED"] = (
    long_tier["University Energy Equiv. – 1B Prompts (MWh)"] * 100
)

# Print published vs derived for the same five models the scoping
# script audited (deterministic seed 42).
rng = np.random.default_rng(42)
candidates = long_tier.dropna(
    subset=["Energy Consumption of 1 Billion Prompts (MWh)"]
).reset_index(drop=True)
sample_idx = rng.choice(len(candidates), size=5, replace=False)
sample = candidates.iloc[sample_idx]

for col_pub, col_der, label in [
    (
        "Energy Consumption of 100 Billion Prompts (MWh)",
        "Energy Consumption of 100 Billion Prompts (MWh) DERIVED",
        "Energy Consumption (MWh) at 100B",
    ),
    (
        "Household Energy Equiv. – 100B Prompts (MWh)",
        "Household Energy Equiv. – 100B Prompts (MWh) DERIVED",
        "Household Energy Equiv. (MWh) at 100B",
    ),
    (
        "University Energy Equiv. – 100B Prompts (MWh)",
        "University Energy Equiv. – 100B Prompts (MWh) DERIVED",
        "University Energy Equiv. (MWh) at 100B",
    ),
]:
    print(f"\n{label} — published vs derived (5 sample models, seed=42):")
    diff = sample[["Model", col_pub, col_der]].copy()
    diff.columns = ["Model", "published", "derived"]
    diff["ratio"] = diff["derived"] / diff["published"]
    print(diff.to_string(index=False))


Energy Consumption (MWh) at 100B — published vs derived (5 sample models, seed=42):
              Model    published       derived  ratio
       GPT-4o (Aug) 22214.537875 222145.378746   10.0
DeepSeek V3 (Azure) 36960.373680 369603.736798   10.0
       Llama 3.1 8B  4430.452863  44304.528628   10.0
        GPT-5 (low) 66508.243675 665082.436750   10.0
       Mistral NeMo  9651.610466  96516.104658   10.0

Household Energy Equiv. (MWh) at 100B — published vs derived (5 sample models, seed=42):
              Model    published       derived  ratio
       GPT-4o (Aug) 20287.249201 202872.492005   10.0
DeepSeek V3 (Azure) 33753.765918 337537.659176   10.0
       Llama 3.1 8B  4046.075674  40460.756738   10.0
        GPT-5 (low) 60738.122078 607381.220777   10.0
       Mistral NeMo  8814.256133  88142.561331   10.0

University Energy Equiv. (MWh) at 100B — published vs derived (5 sample models, seed=42):
              Model  published    derived  ratio
       GPT-4o (Aug)  18.481313 184.81

In [10]:
# Verify no other 100B / 50B columns deviate from the expected ratios
# at the full 66-model long-tier scope. Stop and report if any do.
def ratio_audit_full(df, one_b_col, fifty_col, hundred_col):
    valid = df[df[one_b_col].notna() & (df[one_b_col] != 0)]
    r50 = (valid[fifty_col] / valid[one_b_col]).mean()
    r100 = (valid[hundred_col] / valid[one_b_col]).mean()
    return r50, r100


def find_triplets(cols, b_marker, b_replace_token):
    one_b = {c.replace(f"1{b_marker}", b_replace_token): c for c in cols
             if f"1{b_marker}" in c}
    fifty = {c.replace(f"50{b_marker}", b_replace_token): c for c in cols
             if f"50{b_marker}" in c}
    hundred = {c.replace(f"100{b_marker}", b_replace_token): c for c in cols
               if f"100{b_marker}" in c}
    keys = set(one_b) & set(fifty) & set(hundred)
    return sorted([(one_b[k], fifty[k], hundred[k]) for k in keys])


aggregates = find_triplets(long_tier.columns, " Billion Prompts", "_BP_")
equivalences = find_triplets(long_tier.columns, "B Prompts", "_BP_")
all_triplets = list({tuple(t) for t in aggregates} | {tuple(t) for t in equivalences})

expected_corrected_100b = {
    "Energy Consumption of 100 Billion Prompts (MWh)",
    "Household Energy Equiv. – 100B Prompts (MWh)",
    "University Energy Equiv. – 100B Prompts (MWh)",
}
unexpected_failures = []
for one_b, fifty, hundred in all_triplets:
    r50, r100 = ratio_audit_full(long_tier, one_b, fifty, hundred)
    fail50 = abs(r50 - 50.0) / 50.0 > 0.01
    fail100 = abs(r100 - 100.0) / 100.0 > 0.01
    if fail50:
        unexpected_failures.append((fifty, r50, "50× expected"))
    if fail100 and hundred not in expected_corrected_100b:
        unexpected_failures.append((hundred, r100, "100× expected"))

if unexpected_failures:
    print("NEW unexpected ratio failures (not the three known 100B-MWh columns):")
    for c, r, exp in unexpected_failures:
        print(f"  {c!r}: ratio={r:.3f}, expected {exp}")
    raise AssertionError(
        "new aggregate columns failed the ratio audit beyond the three "
        "100B-MWh-energy columns the user authorised. Stop and surface to chat."
    )
print(
    "all aggregate / equivalence ratios within tolerance except the three "
    "known 100B-MWh-energy columns (Energy Consumption, Household Energy "
    "Equiv., University Energy Equiv.) — corrections applied above."
)

all aggregate / equivalence ratios within tolerance except the three known 100B-MWh-energy columns (Energy Consumption, Household Energy Equiv., University Energy Equiv.) — corrections applied above.


In [11]:
# Build the long-format extract. Six metric families × three scales.
# At 100B scale the energy-MWh metric uses the DERIVED column.
scales = ["1B", "50B", "100B"]

metric_columns = {
    "energy_MWh": {
        "1B":   "Energy Consumption of 1 Billion Prompts (MWh)",
        "50B":  "Energy Consumption of 50 Billion Prompts (MWh)",
        "100B": "Energy Consumption of 100 Billion Prompts (MWh) DERIVED",
    },
    "water_kL": {
        "1B":   "Water Consumption of 1 Billion Prompts (Kiloliter)",
        "50B":  "Water Consumption of 50 Billion Prompts (Kiloliter)",
        "100B": "Water Consumption of 100 Billion Prompts (Kiloliter)",
    },
    "carbon_tCO2e": {
        "1B":   "Carbon Emissions of 1 Billion Prompts (TonsCO2e)",
        "50B":  "Carbon Emissions of 50 Billion Prompts (TonsCO2e)",
        "100B": "Carbon Emissions of 100 Billion Prompts (TonsCO2e)",
    },
    "households_MWh": {
        "1B":   "Household Energy Equiv. – 1B Prompts (MWh)",
        "50B":  "Household Energy Equiv. – 50B Prompts (MWh)",
        "100B": "Household Energy Equiv. – 100B Prompts (MWh) DERIVED",
    },
    "pools_kL": {
        "1B":   "Olympic Swimming Pools Equiv. – 1B Prompts (kL)",
        "50B":  "Olympic Swimming Pools Equiv. – 50B Prompts (kL)",
        "100B": "Olympic Swimming Pools Equiv. – 100B Prompts (kL)",
    },
    "flights_tCO2e": {
        "1B":   "Atlantic Flight Equiv. – 1B Prompts (TonsCO2e)",
        "50B":  "Atlantic Flight Equiv. – 50B Prompts (TonsCO2e)",
        "100B": "Atlantic Flight Equiv. – 100B Prompts (TonsCO2e)",
    },
}

# Per-query value and unit per metric (the per-query column provides
# the chart's drilldown back to the single-prompt magnitude).
per_query_map = {
    "energy_MWh":     ("Mean Combined Energy (Wh)", "Wh"),
    "water_kL":       ("Mean Combined Water (Site & Source, mL)", "mL"),
    "carbon_tCO2e":   ("Mean Combined Carbon (gCO2e)", "gCO2e"),
    "households_MWh": ("Mean Combined Energy (Wh)", "Wh"),
    "pools_kL":       ("Mean Combined Water (Site & Source, mL)", "mL"),
    "flights_tCO2e":  ("Mean Combined Carbon (gCO2e)", "gCO2e"),
}

rows = []
for _, r in long_tier.iterrows():
    for metric, scale_cols in metric_columns.items():
        pq_col, pq_unit = per_query_map[metric]
        if pd.isna(r[pq_col]):
            continue
        for scale in scales:
            value = r[scale_cols[scale]]
            if pd.isna(value):
                continue
            rows.append(
                {
                    "Model": r["Model"],
                    "Company": r["Company"],
                    "scale": scale,
                    "metric": metric,
                    "value": value,
                    "per_query_value": r[pq_col],
                    "per_query_unit": pq_unit,
                }
            )
c5 = pd.DataFrame(rows)
validate_and_write(
    c5,
    TABLEAU / "c5_aggregate_scaling.csv",
    expected_min_rows=700,
    must_have_columns=[
        "Model",
        "Company",
        "scale",
        "metric",
        "value",
        "per_query_value",
        "per_query_unit",
    ],
)

processed\tableau\c5_aggregate_scaling.csv: rows=1116 cols=7


WindowsPath('../data/processed/tableau/c5_aggregate_scaling.csv')

## I. Final summary

In [12]:
outputs = [
    TABLEAU / "c2_model_energy.csv",
    TABLEAU / "c3_regions.csv",
    TABLEAU / "c4_trajectory.csv",
    TABLEAU / "c5_aggregate_scaling.csv",
    TABLEAU / "c6_small_multiples.csv",
    TABLEAU / "c8_flow.csv",
    REFERENCE / "region_coordinates.csv",
]
for p in outputs:
    assert p.exists(), f"missing output: {p}"
    df = pd.read_csv(p)
    rel = p.relative_to(p.parent.parent.parent)
    print(f"\n{rel}")
    print(f"  rows: {len(df)}")
    print(f"  cols: {len(df.columns)}")
    print(f"  columns: {list(df.columns)}")
print(f"\nall {len(outputs)} extracts written.")


processed\tableau\c2_model_energy.csv
  rows: 198
  cols: 6
  columns: ['Model', 'Query Length', 'tier', 'Organization', 'Mean Combined Energy (Wh)', 'is_unmatched']

processed\tableau\c3_regions.csv
  rows: 37
  cols: 12
  columns: ['provider', 'region_name', 'iso3', 'state_abb', 'wue_site', 'cif_site', 'wue_source', 'cif_source', 'grid_carbon_intensity_gco2_kwh', 'water_stress_score', 'us_facility_count_in_state', 'us_facility_sqft_total_in_state']

processed\tableau\c4_trajectory.csv
  rows: 15
  cols: 6
  columns: ['Model', 'Publication date', 'Organization', 'Training compute (FLOP)', 'Parameters', 'Mean Combined Energy (Wh)']

processed\tableau\c5_aggregate_scaling.csv
  rows: 1116
  cols: 7
  columns: ['Model', 'Company', 'scale', 'metric', 'value', 'per_query_value', 'per_query_unit']

processed\tableau\c6_small_multiples.csv
  rows: 36
  cols: 4
  columns: ['Model', 'Query Length', 'tier', 'Mean Combined Energy (Wh)']

processed\tableau\c8_flow.csv
  rows: 5
  cols: 5
  colum